# 3부 자습 노트북 — 배깅과 랜덤 포레스트

본 노트북은 *직접 실행하며* 학습하는 자료이다. 셀을 위에서 아래로 *차례로 실행*하면 3부 이론 교재(`part3_bagging_RF_HARD_이론.md`)의 핵심 코드를 모두 돌려볼 수 있다.

**시리즈에서 본 부의 위치**: 1부(데이터마이닝) 다음으로 진행하는 *트리 가족의 출발점*. 0장에서 결정 트리의 *기초*를 다룬 후, 1~8장에서 *배깅과 랜덤 포레스트*를 본격적으로 다룬다.

**데이터셋**: Ames Housing (2,930 × 82)

## 환경 준비와 데이터 로딩

In [ ]:
# Colab 등에서 처음 한 번만 실행
# !pip install koreanize-matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["axes.unicode_minus"] = False

URL_AMES = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/AmesHousing.csv"

try:
    ames_raw = pd.read_csv(URL_AMES)
except Exception:
    rng = np.random.default_rng(42)
    n = 2900
    ames_raw = pd.DataFrame({
        "Overall Qual": rng.integers(1, 11, n),
        "Gr Liv Area":  rng.integers(500, 4500, n),
        "Year Built":   rng.integers(1900, 2010, n),
        "1st Flr SF":   rng.integers(400, 2000, n),
        "2nd Flr SF":   rng.integers(0, 1500, n),
        "Total Bsmt SF": rng.integers(0, 1500, n),
        "Garage Cars":  rng.integers(0, 4, n),
    })
    ames_raw["SalePrice"] = (
        50000 + ames_raw["Overall Qual"] * 25000
        + ames_raw["Gr Liv Area"] * 60 + rng.normal(0, 20000, n)
    ).astype(int)

print(f"Ames: {ames_raw.shape}")

# 1부 표준 전처리
def prepare_ames(df_in):
    df = df_in.copy()
    df = df.drop(columns=[c for c in ["Order", "PID"] if c in df.columns])
    for c in ["Pool QC", "Misc Feature", "Alley", "Fence", "Fireplace Qu",
              "Garage Qual", "Garage Cond", "Garage Finish", "Garage Type",
              "Bsmt Qual", "Bsmt Cond"]:
        if c in df.columns:
            df[c] = df[c].fillna("None")
    num = df.select_dtypes("number").columns
    df[num] = df[num].fillna(df[num].median())
    df = df[df["Gr Liv Area"] < 4000].copy()
    if all(c in df.columns for c in ["1st Flr SF", "2nd Flr SF", "Total Bsmt SF"]):
        df["Total SF"] = df["1st Flr SF"] + df["2nd Flr SF"] + df["Total Bsmt SF"]
    return df

ames = prepare_ames(ames_raw)
y_ames = np.log1p(ames["SalePrice"])
X_ames = ames.select_dtypes("number").drop(columns=["SalePrice"])
print(f"전처리 후: X {X_ames.shape}, y {y_ames.shape}")

---
## 0장 결정 트리 — 질문의 사슬로 답을 찾는 모델

결정 트리는 *if-else 질문의 사슬*이다. 각 분할은 *불순도가 가장 줄어드는 방향*으로 선택된다.

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

# 작은 트리 학습
tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tree.fit(X_ames, y_ames)

fig, ax = plt.subplots(figsize=(20, 8), dpi=100)
plot_tree(tree,
          feature_names=X_ames.columns,
          filled=True, rounded=True,
          fontsize=10, ax=ax)
plt.tight_layout()
plt.show()

print(f"\n루트 분할 변수: {X_ames.columns[tree.tree_.feature[0]]}")
print(f"루트 임계값:    {tree.tree_.threshold[0]:.2f}")
print(f"잎 수:         {tree.get_n_leaves()}")

---
## 1장 한 그루 트리의 흔들림과 평균의 안정

단일 트리는 *학습 데이터에 매우 민감*하다. 약간 다른 데이터로 학습하면 *완전히 다른 트리*가 만들어진다.

In [ ]:
from sklearn.model_selection import train_test_split

# 같은 데이터에서 random_state만 바꿔서 여러 트리 학습 (부트스트랩 시뮬레이션)
rng = np.random.default_rng(42)

# 5개 트리의 첫 분할 변수와 임계값 비교
print("같은 데이터에서 시드만 바꾼 5개 트리의 첫 분할:")
print()
for seed in range(5):
    # 부트스트랩 (복원 추출)
    idx = rng.choice(len(X_ames), size=len(X_ames), replace=True)
    X_boot, y_boot = X_ames.iloc[idx], y_ames.iloc[idx]
    
    t = DecisionTreeRegressor(max_depth=5, random_state=seed)
    t.fit(X_boot, y_boot)
    feat = X_ames.columns[t.tree_.feature[0]]
    thresh = t.tree_.threshold[0]
    print(f"  시드 {seed}: 첫 분할 '{feat}' <= {thresh:.2f}")

print()
print("→ 부트스트랩만으로도 첫 분할 변수가 흔들린다. 이게 단일 트리의 '분산'.")

---
## 2장 복원 추출과 부트스트랩 표본

원본 데이터에서 *같은 크기 n*을 *복원 추출*한 표본이 부트스트랩 표본이다. 평균적으로 *63.2%의 원본 행*이 들어간다.

In [ ]:
n = 1000
rng = np.random.default_rng(42)

# 부트스트랩 표본 1개 생성
idx = rng.choice(n, size=n, replace=True)
unique_in_boot = len(set(idx))
print(f"원본 크기: {n}")
print(f"부트스트랩 표본 크기: {len(idx)}")
print(f"표본에 포함된 유일 행 수: {unique_in_boot}")
print(f"포함 비율: {unique_in_boot/n*100:.1f}%  (이론값 약 63.2%)")
print()

# 여러 표본 평균
ratios = []
for _ in range(100):
    idx = rng.choice(n, size=n, replace=True)
    ratios.append(len(set(idx)) / n)
print(f"100회 시뮬레이션 평균 포함 비율: {np.mean(ratios)*100:.2f}%")
print(f"  → 이론값 1 - (1-1/n)^n ≈ {(1 - (1 - 1/n)**n)*100:.2f}%")

---
## 3장 분산 감소의 수학 — 평균의 위력

T개의 트리를 *평균*하면 *분산이 1/T로 줄어든다* (트리들이 독립일 때).

실제로 T를 늘려 가며 R²가 어떻게 변하는지 본다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

print(f"{'T (트리 수)':>10s}  {'CV R²':>10s}")
print("-" * 24)
for n_est in [1, 5, 10, 25, 50, 100, 200]:
    rf = RandomForestRegressor(n_estimators=n_est, random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    print(f"{n_est:>10d}  {r2:>10.4f}")

print("\n→ T가 커질수록 R² 향상. 그러나 100그루 이후로는 거의 평탄.")

---
## 4장 무작위 변수 선택 — 트리 간 상관 깨기

부트스트랩만으로는 트리들이 *비슷하게 학습*된다. RF는 *각 분할에서 일부 변수만* 후보로 평가하여 *트리 다양성*을 더한다.

In [ ]:
# max_features에 따른 R² 변화
import math

n_features = X_ames.shape[1]
print(f"전체 변수 수: {n_features}")
print()

print(f"{'max_features':>15s}  {'CV R²':>10s}")
print("-" * 28)
for mf in ["sqrt", "log2", 0.3, 0.5, 1.0]:
    rf = RandomForestRegressor(n_estimators=100, max_features=mf,
                                random_state=42, n_jobs=-1)
    r2 = cross_val_score(rf, X_ames, y_ames, cv=3, scoring="r2", n_jobs=-1).mean()
    if isinstance(mf, str):
        actual = math.ceil(math.sqrt(n_features)) if mf == "sqrt" else math.ceil(math.log2(n_features))
        label = f"{mf} (={actual})"
    else:
        label = f"{mf}"
    print(f"{label:>15s}  {r2:>10.4f}")

print("\n→ sqrt(d) 가 회귀에서는 기본. d/3도 자주 쓰임.")

---
## 5장 OOB 점수 — 외부 검증 없는 일반화 추정

부트스트랩에 *포함되지 않은 약 36.8% 샘플*(out-of-bag, OOB)을 *검증 데이터*로 사용. 별도의 train/test split 없이도 일반화 성능을 추정할 수 있다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_oob = RandomForestRegressor(
    n_estimators=200,
    oob_score=True,                   # OOB 점수 켜기
    random_state=42,
    n_jobs=-1
)
rf_oob.fit(X_ames, y_ames)

print(f"OOB R²:        {rf_oob.oob_score_:.4f}")
print(f"학습 데이터 R²: {rf_oob.score(X_ames, y_ames):.4f}")
print()
print("→ OOB R²가 실제 일반화 성능에 가까운 추정. 학습 R²는 과대평가됨.")

---
## 6장 변수 중요도 — 두 가지 계산 방법

(1) **불순도 감소** (`feature_importances_`) — 빠르지만 *고cardinality 변수에 편향*
(2) **순열 중요도** (`permutation_importance`) — 느리지만 정확

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_ames, y_ames)

# 방법 1: 불순도 감소
imp_impurity = pd.Series(rf.feature_importances_, index=X_ames.columns)
print("불순도 감소 중요도 상위 10:")
print(imp_impurity.nlargest(10).round(4))

In [ ]:
from sklearn.inspection import permutation_importance

# 방법 2: 순열 중요도 (속도를 위해 n_repeats=5)
result = permutation_importance(rf, X_ames, y_ames,
                                  n_repeats=5, random_state=42, n_jobs=-1)
imp_perm = pd.Series(result.importances_mean, index=X_ames.columns)

print("순열 중요도 상위 10:")
print(imp_perm.nlargest(10).round(4))

In [ ]:
# 두 중요도 비교 시각화
top10 = imp_impurity.nlargest(10).index
df_imp = pd.DataFrame({
    "불순도": imp_impurity[top10] / imp_impurity[top10].max(),  # 정규화
    "순열":   imp_perm[top10] / imp_perm[top10].max()
})

fig, ax = plt.subplots(figsize=(10, 6))
df_imp.plot(kind="barh", ax=ax, color=["#1F3A5F", "#C0392B"])
ax.invert_yaxis()
ax.set_xlabel("중요도 (정규화)")
ax.set_title("두 가지 변수 중요도 비교 — 상위 10개")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

---
## 7장 Ames 데이터에서 랜덤 포레스트의 동작

RF의 최종 성능과 매개변수 효과를 종합 정리.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

models = {
    "단일 트리 (기본)":      DecisionTreeRegressor(random_state=42),
    "RF 10그루":             RandomForestRegressor(n_estimators=10, random_state=42, n_jobs=-1),
    "RF 100그루":            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "RF 500그루":            RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1),
}

print(f"{'모델':<25s}  {'CV R²':>10s}")
print("-" * 38)
for name, m in models.items():
    r2 = cross_val_score(m, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<25s}  {r2:>10.4f}")

---
## 8장 배깅의 한계 — 부스팅으로의 다리

배깅은 *모든 트리가 독립적으로 동시 학습*한다. *부스팅*은 *순차적으로 학습*해서 *이전 트리의 오류를 보정*. 다음 부에서 본다.

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor

models = {
    "RF (배깅)":            RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "AdaBoost (부스팅)":     AdaBoostRegressor(n_estimators=100, random_state=42),
    "GBM (부스팅)":          GradientBoostingRegressor(n_estimators=100, random_state=42),
}

print(f"{'모델':<25s}  {'CV R²':>10s}")
print("-" * 38)
for name, m in models.items():
    r2 = cross_val_score(m, X_ames, y_ames, cv=5, scoring="r2", n_jobs=-1).mean()
    print(f"{name:<25s}  {r2:>10.4f}")

print("\n→ 부스팅이 약간 더 좋다. 4부에서 AdaBoost, 5부에서 GBM을 다룬다.")

---
## 마무리

본 노트북에서 *직접 실행*한 8가지 RF 도구를 한 표로 정리한다.

| 장 | 핵심 도구 |
|---|---|
| 0장 결정 트리 | `DecisionTreeRegressor`, `plot_tree` |
| 1장 트리 흔들림 | 부트스트랩 시뮬레이션 |
| 2장 부트스트랩 | `np.random.choice(..., replace=True)` |
| 3장 분산 감소 | `n_estimators` 효과 |
| 4장 무작위 변수 | `max_features="sqrt"` |
| 5장 OOB 점수 | `oob_score=True` |
| 6장 변수 중요도 | `feature_importances_`, `permutation_importance` |
| 7장 RF 종합 | `RandomForestRegressor` 매개변수 |
| 8장 부스팅 | AdaBoost, GBM과 비교 |

### 다음 단계

2부로 이동하여 **결정 트리 심화**(`part2_tree_advanced_HARD_자습노트북.ipynb`)를 자습한다. 또는 4부로 이동하여 *부스팅의 첫 사례 AdaBoost*를 다룬다.